In [1]:
from pydantic import BaseModel, Field
import ollama
from utils import *

In [2]:
system_prompt = """
You are an elite Olympic discus coach and biomechanist.

You have been given a sequence of images showing a complete discus throw from beginning to end.

Evaluate the thrower's technique using these criteria:
- balance and posture
- rotation mechanics
- footwork
- hip and shoulder separation
- release position
- follow-through

Be strict but fair. Rate the throw from 1-10 where:
1 = very poor form
5 = average high school athlete
8 = strong collegiate athlete
10 = world class technique

Give 2-4 sentences of specific, actionable feedback.
"""


In [3]:
system_prompt = """
You are a function that evaluates discus throw images.

Your job is to return one JSON object only.

Rules:
1. Output must be valid JSON.
2. Do not output markdown.
3. Do not output explanations.
4. Do not output text before or after the JSON.
5. Use this exact schema:
{
  "score": integer from 1 to 10,
  "feedback": string
}
6. "feedback" must be 2 to 4 sentences.
7. Be specific and actionable.
8. If uncertain, still return valid JSON and make the best judgment from the images.

Scoring rubric:
- 1 to 3: major posture, balance, or release problems
- 4 to 5: below-average throw with clear technical issues
- 6 to 7: decent throw with some useful mechanics but important flaws
- 8 to 9: strong throw with good technical quality
- 10: elite/world-class technique

Judge the throw using:
- balance and posture
- rotation mechanics
- footwork
- hip and shoulder separation
- release position
- follow-through

Return JSON only.
"""

In [4]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles

MODEL_PATH = "pose_landmarker.task"

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    min_pose_detection_confidence=0.1,
    min_pose_presence_confidence=0.1,
    min_tracking_confidence=0.1
)
detector = vision.PoseLandmarker.create_from_options(options)



I0000 00:00:1775668546.134046   89665 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775668546.169033   89668 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775668546.182754   89677 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [5]:
frames = sample_frames_from_mp4("throw12.mp4",7)
overs=pose_overlays_from_frames(frames,detector)
perfect_paths=write_temp_images_for_llm(overs,"prefectfewshot")

frames = sample_frames_from_mp4("videoplayback-4.mp4",6)
overs=pose_overlays_from_frames(frames,detector)
scoresix_paths=write_temp_images_for_llm(overs,"score6fewshot")

W0000 00:00:1775668546.846746   89671 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [6]:
system=[
        {
            "role": "system",
            "content": system_prompt
        }]

In [19]:
fewshots=[{
        "role": "user",
        "content": "Please evaluate this discus throw.",
        "images": perfect_paths
    },
    {
        "role": "assistant",
        "content": '{"score":10,"feedback":"Excellent balance, form, and timing throughout the throw. Strong hip-to-shoulder separation and a high release. Stay slightly taller through the finish."}'
    },
        {
            "role": "user",
            "content": "Please evaluate this discus throw.",
            "images": scoresix_paths
        },
        {
            "role": "assistant",
            "content":'{"score": 6,"feedback": "Foot stance is much wider than hips. Athlete upon leaving the wind up immediately looks over left shoulder, which puts the athlete off balance and never winds the spring in the torso properly because of it.  The left side leads the whole way.   Discus is also not held at or near shoulder height, dips coming around to sprint and again to release."}'
        }
     ]

In [8]:
fewshots[2:]

[{'role': 'user',
  'content': 'Please evaluate this discus throw.',
  'images': ['score6fewshot/frame_0.png',
   'score6fewshot/frame_1.png',
   'score6fewshot/frame_2.png',
   'score6fewshot/frame_3.png',
   'score6fewshot/frame_4.png',
   'score6fewshot/frame_5.png']},
 {'role': 'assistant',
  'content': '{"score": 6,"feedback": "Foot stance is much wider than hips. Athlete upon leaving the wind up immediately looks over left shoulder, which puts the athlete off balance and never winds the spring in the torso properly because of it.  The left side leads the whole way.   Discus is also not held at or near shoulder height, dips coming around to sprint and again to release."}'}]

In [26]:
## from pydantic import BaseModel, Field
import ollama

class ThrowCritique(BaseModel):
    score: int = Field(ge=1, le=10)
    feedback: str = Field(
        min_length=20,
        max_length=500,
        description="Brief coaching feedback"
    )


def critique_throw(
    image_paths: list,
    mod: str = "gemma3:12b",
    temp: float = 0.0
    ):
    messages =system+fewshots +[{
    "role": "user",
    "content": "Please evaluate this discus throw.",
    "images": image_paths
        }]

    resp = ollama.chat(
        model=mod,
        messages=messages,
        format=ThrowCritique.model_json_schema(),
        options={"temperature": temp}
    )

    raw = resp["message"]["content"]
    return ThrowCritique.model_validate_json(raw)

In [27]:
frames = sample_frames_from_mp4("videoplayback.mp4",10)
overs=pose_overlays_from_frames(frames,detector)
img_paths=write_temp_images_for_llm(overs)
crit=critique_throw(image_paths=img_paths)
crit

ThrowCritique(score=8, feedback="Sandra Perkovic's technique is excellent. She demonstrates a very powerful and efficient rotation, with a strong wind-up and a full extension during the release. Her body positioning and balance are superb, allowing her to generate maximum force. The discus is held high and released at the optimal point. The throw is very clean and consistent, which is reflected in her winning performance at the World Championships.")